# 05 · 3Sum & 4Sum (Hashing and Two Pointers)

**3Sum:** find all *unique* triplets `[a, b, c]` with `a + b + c == 0`.
**4Sum:** find all *unique* quadruplets that add up to a `target`.

| Problem | Brute force | Hashing | **Sort + two pointers** |
|---|---|---|---|
| 3Sum | O(n³) | O(n²) time, O(n) space | **O(n²)**, no extra set |
| 4Sum | O(n⁴) | O(n³) time, O(n) space | **O(n³)** |

The big idea for the optimal approach: **sort the array**, fix the first one (or two) numbers with loops, then use two pointers `j` (left) and `k` (right) to find the rest.
- sum too small → move the left pointer right (`j += 1`)
- sum too big → move the right pointer left (`k -= 1`)
- sum matches → record it, move both, and **skip duplicate values**.

## 1. 3Sum

### Approach 1 – Brute force: three loops
Sort each triplet and store it in a `set` of tuples so duplicates collapse automatically.
Time O(n³).

In [1]:
nums = [-1, 0, 1, 2, -1, -4]
n = len(nums)

found = set()
for i in range(n):
    for j in range(i + 1, n):
        for k in range(j + 1, n):
            if nums[i] + nums[j] + nums[k] == 0:
                found.add(tuple(sorted([nums[i], nums[j], nums[k]])))
brute_3sum = sorted(found)
print(brute_3sum)

[(-1, -1, 2), (-1, 0, 1)]


### Approach 2 – Hashing: two loops
Fix `nums[i]` and `nums[j]`. The third number must be `-(nums[i] + nums[j])`. Instead of a third loop, look for it in a set of values we have already passed **between** `i` and `j`.
Time O(n²), Space O(n).

In [2]:
found = set()
for i in range(n):
    seen = set()                                 # new set for every i
    for j in range(i + 1, n):
        third = -(nums[i] + nums[j])
        if third in seen:
            found.add(tuple(sorted([nums[i], nums[j], third])))
        seen.add(nums[j])
hash_3sum = sorted(found)
print(hash_3sum)

[(-1, -1, 2), (-1, 0, 1)]


### Approach 3 – Sort + two pointers (optimal)

**Duplicate handling**
- Skip an `i` if `nums[i] == nums[i-1]` (we already used this value as the first number).
- After finding a triplet, move `j` and `k` past equal neighbours.

> 🐞 **Fix:** the earlier version started the left pointer with `j = 1 + 1` (always 2!). It must be `j = i + 1`. Also, the input must be **sorted** *before* the loop.

In [3]:
def three_sum(nums):
    nums = sorted(nums)                          # sorting is required
    n = len(nums)
    ans = []
    for i in range(n):
        if i > 0 and nums[i] == nums[i - 1]:     # skip duplicate first numbers
            continue
        j, k = i + 1, n - 1
        while j < k:
            total = nums[i] + nums[j] + nums[k]
            if total < 0:
                j += 1                           # need a bigger sum
            elif total > 0:
                k -= 1                           # need a smaller sum
            else:
                ans.append([nums[i], nums[j], nums[k]])
                j += 1
                k -= 1
                while j < k and nums[j] == nums[j - 1]:   # skip duplicate j values
                    j += 1
                while j < k and nums[k] == nums[k + 1]:   # skip duplicate k values
                    k -= 1
    return ans

print(three_sum([-1, 0, 1, 2, -1, -4]))                              # [[-1, -1, 2], [-1, 0, 1]]
print(three_sum([-2, -2, -2, -1, -1, -1, 0, 0, 0, 2, 2, 2, 2]))      # lots of duplicates

# Cross-check the three approaches on the same input
assert [list(t) for t in brute_3sum] == [list(t) for t in hash_3sum] == sorted(three_sum([-1, 0, 1, 2, -1, -4]))
print("✅ all three 3Sum approaches agree")

[[-1, -1, 2], [-1, 0, 1]]
[[-2, 0, 2], [-1, -1, 2], [0, 0, 0]]
✅ all three 3Sum approaches agree


## 2. 4Sum

### Approach 1 – Brute force: four loops (O(n⁴))

In [4]:
nums = [-2, -2, -2, -1, -1, -1, 0, 0, 0, 2, 2, 2, 2]
target = 0
n = len(nums)

found = set()
for i in range(n):
    for j in range(i + 1, n):
        for k in range(j + 1, n):
            for l in range(k + 1, n):
                if nums[i] + nums[j] + nums[k] + nums[l] == target:
                    found.add(tuple(sorted([nums[i], nums[j], nums[k], nums[l]])))
brute_4sum = sorted(found)
print(brute_4sum)

[(-2, -2, 2, 2), (-2, 0, 0, 2), (-1, -1, 0, 2)]


### Approach 2 – Hashing: three loops (O(n³))
Fix `i`, `j`, `k`; look for the fourth value `target − (a + b + c)` in a set.

> 🐞 **Bug fixed:** the `seen` set must be **re-created for every pair `(i, j)`** and hold only elements *after* `j`.
> If the set is only reset per `i`, it can contain `nums[j]` itself (or other earlier values), so the same element gets used twice. On `[-3, -1, 0, 2, 4, 5]` with target `1` the old code wrongly returned `[-3, 0, 2, 2]` and `[-3, 0, 0, 4]`.

In [5]:
def four_sum_hash(nums, target):
    n = len(nums)
    found = set()
    for i in range(n):
        for j in range(i + 1, n):
            seen = set()                         # ← fresh set for each (i, j) pair
            for k in range(j + 1, n):
                fourth = target - (nums[i] + nums[j] + nums[k])
                if fourth in seen:
                    found.add(tuple(sorted([nums[i], nums[j], nums[k], fourth])))
                seen.add(nums[k])
    return sorted(found)

print(four_sum_hash([-2, -2, -2, -1, -1, -1, 0, 0, 0, 2, 2, 2, 2], 0))
print(four_sum_hash([-3, -1, 0, 2, 4, 5], 1))    # only [-3, -1, 0, 5]

[(-2, -2, 2, 2), (-2, 0, 0, 2), (-1, -1, 0, 2)]
[(-3, -1, 0, 5)]


### Approach 3 – Sort + two pointers (optimal for this problem)
Two nested loops fix `i` and `j`; then the same two-pointer sweep as 3Sum runs on `k` and `l`. Skip duplicates at **all four** positions.

In [6]:
def four_sum(nums, target):
    nums = sorted(nums)
    n = len(nums)
    ans = []
    for i in range(n):
        if i > 0 and nums[i] == nums[i - 1]:             # skip duplicate i
            continue
        for j in range(i + 1, n):
            if j > i + 1 and nums[j] == nums[j - 1]:     # skip duplicate j
                continue
            k, l = j + 1, n - 1
            while k < l:
                total = nums[i] + nums[j] + nums[k] + nums[l]
                if total == target:
                    ans.append([nums[i], nums[j], nums[k], nums[l]])
                    k += 1
                    l -= 1
                    while k < l and nums[k] == nums[k - 1]:   # skip duplicate k
                        k += 1
                    while k < l and nums[l] == nums[l + 1]:   # skip duplicate l
                        l -= 1
                elif total < target:
                    k += 1
                else:
                    l -= 1
    return ans

print(four_sum([-3, -1, 0, 2, 4, 5], 1))                                 # [[-3, -1, 0, 5]]
print(four_sum([-2, -2, -2, -1, -1, -1, 0, 0, 0, 2, 2, 2, 2], 0))

# Cross-check
big = [-2, -2, -2, -1, -1, -1, 0, 0, 0, 2, 2, 2, 2]
assert [list(t) for t in brute_4sum] == [list(t) for t in four_sum_hash(big, 0)] == sorted(four_sum(big, 0))
print("✅ all 4Sum approaches agree")

[[-3, -1, 0, 5]]
[[-2, -2, 2, 2], [-2, 0, 0, 2], [-1, -1, 0, 2]]
✅ all 4Sum approaches agree


---
### ✅ Key takeaways
- **Sort first**, then two pointers; sorting makes duplicates adjacent and gives the "too small / too big" signal.
- Each extra number you fix with a loop costs another factor of `n`: 2Sum O(n) → 3Sum O(n²) → 4Sum O(n³).
- In the hashing version, the lookup set must contain **only valid, not-yet-used** elements.